In [1]:
# Imports
import os as os
import netCDF4 as nc
import glob
import time
import GRUAN_fxn_lib as lib


In [2]:
#######################
years = ['2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021'] # Years of interest
matching_data = []

# Get all GRUAN radiosonde files
files = []
for year in years:
    files.extend(glob.glob('/home/chinahg/GCresearch/GRUAN_sondes/ftp.ncdc.noaa.gov/pub/data/gruan/processing/level2/RS92-GDP/version-002/'+'/**/'+year+'/*.nc', recursive=True))

files.sort()
num_files = len(files)

filenames = [os.path.basename(file) for file in files]
site_names = [os.path.basename(file).split('-')[0] for file in files]
datetimes = [os.path.basename(file).split('_')[4] for file in files]

for i in range(num_files):
    matching_data.append([filenames[i], site_names[i], datetimes[i], files[i]])

In [3]:
for i in range(num_files):
    path2file = files[i]

    # Read file in the location of interest
    nc_GRUAN = nc.Dataset(path2file, 'r', format='NETCDF4_CLASSIC')
    RH_w = nc_GRUAN.variables['rh'][:] # [unitless, not %]
    T = nc_GRUAN.variables['temp'][:] # [K]

    # Close the read version of the file
    nc_GRUAN.close()

    # Open the file in write mode
    nc_GRUAN = nc.Dataset(path2file, 'a', format='NETCDF4_CLASSIC')

    P_sat_w = lib.compute_Psat_w(T) # [Pa]
    P_sat_i =  lib.compute_Psat_i(T) # [Pa]
    RH_i = RH_w*(P_sat_w/P_sat_i) # [unitless]

     # Define the new variable in the NetCDF file
    if 'rh_i' not in nc_GRUAN.variables.keys():
        print('Creating new variable for RH_i, file index:', i)
        rh_i = nc_GRUAN.createVariable('rh_i', 'f4', ('time',))
        rh_i.units = 'unitless'
        rh_i.long_name = 'Relative Humidity with respect to ice'
    
    # Assign the RH_i data to the new variable
    nc_GRUAN.variables['rh_i'][:] = RH_i

    #Close the NetCDF file
    nc_GRUAN.close()

In [4]:
print("Finished GRUAN Processing!")

Finished GRUAN Processing!
